# перенос импульса по кольцу

Решаем уравнение

$$u_t+c u_x=0,\qquad x\in[0,1),\qquad c=1$$

с периодическими граничными условиями. Точное решение — сдвиг начального профиля:

$$u(x,t)=u_0\bigl((x-ct)\bmod1\bigr).$$

Один оборот занимает время $T=1$. Используем узлы $x_j=j\Delta x$, где $\Delta x=1/N$ и $j=0,\ldots,N-1$.
Точку $x=1$ в расчётную сетку не включаем.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path
from tempfile import TemporaryDirectory

labels = {'upwind': 'схема против потока', 'LW': 'схема Лакса–Вендроффа'}
colors = {'upwind': 'tab:blue', 'LW': 'tab:orange'}
styles = {'upwind': '-', 'LW': '--'}

## разностные схемы

При $c>0$ схема против потока имеет вид

$$u_j^{n+1}=(1-C_n)u_j^n+C_nu_{j-1}^n.$$

Для схемы Лакса–Вендроффа:

$$u_j^{n+1}=u_j^n-\frac{C_n}{2}(u_{j+1}^n-u_{j-1}^n)
+\frac{C_n^2}{2}(u_{j+1}^n-2u_j^n+u_{j-1}^n).$$

Индексы периодические, $C_n=c\Delta t_n/\Delta x$.
Основной шаг равен $\Delta t=C\Delta x/c$. До заданного времени при необходимости сокращаем только последний шаг и пересчитываем его $C_n$.
Расчёты до $t=1$ и $t=5$ выполняем отдельно, чтобы промежуточный вывод не менял шаги.

Вычисляем ошибку, количество примеси и экстремумы:

$$E_1(t)=\Delta x\sum_j|u_j(t)-u(x_j,t)|,\qquad
M(t)=\Delta x\sum_j u_j(t),\qquad u_{\min}(t),\ u_{\max}(t).$$

In [ ]:
def initial(x, profile):  # начальный профиль в периодической области
    x = np.asarray(x)
    if profile == 'smooth':
        return 1 + 0.5*np.sin(2*np.pi*x)
    # Устраняем погрешность округления координаты на границах импульса.
    z = np.round(x % 1, 12) % 1
    return ((z >= 0.2) & (z < 0.4)).astype(float)


def exact(x, t, profile):  # точное решение при c = 1
    return initial((np.asarray(x) - t % 1) % 1, profile)


def upwind(u, C):  # один шаг схемы против потока
    return (1-C)*u + C*np.roll(u, 1)


def lax_wendroff(u, C):  # один шаг схемы Лакса–Вендроффа
    left, right = np.roll(u, 1), np.roll(u, -1)
    return C*(C+1)/2*left + (1-C*C)*u + C*(C-1)/2*right


methods = {'upwind': upwind, 'LW': lax_wendroff}


def solve(N, C, t_end, profile):  # обе схемы на одной сетке с сохранением истории
    x = np.arange(N) / N
    dx = 1 / N
    dt = C*dx
    n = int(np.ceil(t_end/dt - 1e-12))
    times = np.arange(n+1)*dt
    times[-1] = t_end
    courant = np.full(n, C, dtype=float)
    courant[-1] = (t_end - (n-1)*dt) / dx
    if np.isclose(courant[-1], C, rtol=0, atol=1e-12):
        courant[-1] = C

    reference = np.array([exact(x, t, profile) for t in times])
    result = {'x': x, 't': times, 'C_steps': courant, 'profile': profile}
    for name, step in methods.items():
        history = np.empty((n+1, N))
        history[0] = initial(x, profile)
        for k, C_step in enumerate(courant):
            history[k+1] = step(history[k], C_step)
        result[name] = {
            'u': history,
            'error': dx*np.sum(abs(history-reference), axis=1),
            'mass': dx*np.sum(history, axis=1),
            'minimum': np.min(history, axis=1),
            'maximum': np.max(history, axis=1),
        }
    return result

In [ ]:
def plot_profile(ax, run, name, index=-1):  # численный профиль с замыканием кольца на рисунке
    x = np.r_[run['x'], 1.0]
    u = run[name]['u'][index]
    return ax.plot(x, np.r_[u, u[0]], color=colors[name], linestyle=styles[name],
                   linewidth=1.7, label=labels[name],
                   drawstyle='steps-post' if run['profile'] == 'pulse' else 'default')[0]


def pulse_line(t):  # точные положения обоих фронтов, включая переход через границу
    edges = np.round((np.array([0.2, 0.4]) + t) % 1, 12) % 1
    bounds = np.unique(np.r_[0.0, edges, 1.0])
    middle = (bounds[:-1] + bounds[1:]) / 2
    values = exact(middle, t, 'pulse')
    x = np.column_stack((bounds[:-1], bounds[1:])).ravel()
    return x, np.repeat(values, 2)

## гладкий профиль

$$u_0(x)=1+\frac12\sin(2\pi x),\qquad C=0.8.$$

Сравниваем обе схемы с точным решением после одного и пяти оборотов на сетках $N=50,100,200$.
При каждом $N$ сохраняем основное число Куранта $0.8$. Например, при $N=50$ до $t=1$ выполняются 62 полных шага и один сокращённый с $C_n=0.4$.

In [ ]:
sizes = [50, 100, 200]
end_times = [1.0, 5.0]
smooth_runs = {(N, t): solve(N, 0.8, t, 'smooth') for N in sizes for t in end_times}
x_fine = np.linspace(0, 1, 1001)  # только для отрисовки точного решения

fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True, sharey=True, constrained_layout=True)
for row, N in enumerate(sizes):
    for col, t in enumerate(end_times):
        ax = axes[row, col]
        run = smooth_runs[N, t]
        for name in methods:
            plot_profile(ax, run, name)
        ax.plot(x_fine, exact(x_fine, t, 'smooth'), ':', color='black', label='точное решение')
        ax.set(xlabel='x', ylabel='u', xlim=(0, 1), ylim=(0.44, 1.56), title=f'N = {N}, t = {t:g}')
        ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
plt.show()

print(' t    N      E₁, upwind      E₁, Л–В       M, upwind       M, Л–В')
for t in end_times:
    for N in sizes:
        run = smooth_runs[N, t]
        print(f"{t:2g}  {N:3d}     {run['upwind']['error'][-1]:.5e}    {run['LW']['error'][-1]:.5e}    {run['upwind']['mass'][-1]:.10f}    {run['LW']['mass'][-1]:.10f}")

In [ ]:
dx_values = 1 / np.array(sizes)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, t in zip(axes, end_times):
    for name in methods:
        errors = np.array([smooth_runs[N, t][name]['error'][-1] for N in sizes])
        ax.loglog(dx_values, errors, 'o-', color=colors[name], label=labels[name])
        order = np.log2(errors[:-1] / errors[1:])
        print(f't = {t:g}, {labels[name]}: p = {order[0]:.2f}, {order[1]:.2f}')
    ax.set(xlabel='Δx', ylabel='E₁', title=f't = {t:g}')
    ax.set_xticks(dx_values, labels=['1/50', '1/100', '1/200'])
    ax.xaxis.set_minor_locator(NullLocator())
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.show()

При сгущении сетки ошибка уменьшается. Для схемы против потока измеренный порядок близок к первому, для схемы Лакса–Вендроффа — ко второму.
Порядок оцениваем как $p=\log_2(E_1(\Delta x)/E_1(\Delta x/2))$.
После пяти оборотов при $N=200$ ошибки составляют примерно $0.0299$ и $0.000592$ соответственно.

В решении по схеме против потока заметно уменьшается амплитуда синусоиды — это численная диффузия.
Для схемы Лакса–Вендроффа искажение на этих сетках существенно меньше.

## количество примеси

Начальное количество примеси равно $M(0)=1$. На графике показываем $M(t)-M(0)$, чтобы были видны малые изменения.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True, constrained_layout=True)
for ax, N in zip(axes, sizes):
    run = smooth_runs[N, 5.0]
    for name in methods:
        mass = run[name]['mass']
        ax.plot(run['t'], mass-mass[0], color=colors[name], linestyle=styles[name], label=labels[name])
        print(f"N = {N}, {labels[name]}: max |M(t) − M(0)| = {np.max(abs(mass-mass[0])):.2e}")
    ax.set(xlabel='t', ylabel='M(t) − M(0)', title=f'N = {N}')
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0), useOffset=False)
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
plt.show()

Для обеих схем сумма разностных добавок по периодической сетке равна нулю. Поэтому количество примеси сохраняется с точностью округления.
Это не означает, что распределение найдено правильно: на тех же расчётах видны сглаживание профиля и ненулевая ошибка $E_1$.

## прямоугольный импульс

$$u_0(x)=\begin{cases}1,&0.2\leq x<0.4,\\0,&\text{вне этого интервала}.\end{cases}$$

Берём $N=200$ и $C=0.5,0.8,1$. Отрицательные значения и выбросы выше единицы сохраняем.
Ниже показаны полные профили при $t=5$, затем оба фронта при $t=1$ в одинаковых масштабах.

In [ ]:
N_pulse = 200
stable_C = [0.5, 0.8, 1.0]
stable_runs = {C: solve(N_pulse, C, 5.0, 'pulse') for C in stable_C}
front_runs = {C: solve(N_pulse, C, 1.0, 'pulse') for C in stable_C}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True, constrained_layout=True)
for ax, C in zip(axes, stable_C):
    for name in methods:
        plot_profile(ax, stable_runs[C], name)
    ax.plot(*pulse_line(5), ':', color='black', label='точное решение')
    ax.set(xlabel='x', ylabel='u', xlim=(0, 1), ylim=(-0.35, 1.35), title=f'C = {C:g}, t = 5')
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
plt.show()

fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharey=True, constrained_layout=True)
for row, C in enumerate(stable_C):
    for col, (edge, title) in enumerate([(0.2, 'левый фронт'), (0.4, 'правый фронт')]):
        ax = axes[row, col]
        for name in methods:
            plot_profile(ax, front_runs[C], name)
        ax.plot(*pulse_line(1), ':', color='black', label='точное решение')
        ax.set(xlabel='x', ylabel='u', xlim=(edge-0.1, edge+0.1), ylim=(-0.3, 1.3),
               title=f'{title}, C = {C:g}, t = 1')
        ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
plt.show()

При $0<C<1$ в схеме против потока новое значение — взвешенное среднее двух соседних значений. Поэтому новых экстремумов не возникает, но фронты размываются.

У схемы Лакса–Вендроффа коэффициент при $u_{j+1}^n$ равен $C(C-1)/2$ и отрицателен при $0<C<1$.
Сохранение диапазона $[0,1]$ не обеспечивается: около фронтов появляются колебания, отрицательные значения и выбросы выше единицы.
Второй порядок точности относится к гладкому решению; на разрыве используемое разложение по производным неприменимо.

## сгущение сетки

Сравниваем $N=50,100,200,400$ при одном времени $t=5$, сохраняя выбранное $C$.
В таблице приведены ошибка и оба экстремума для всех трёх чисел Куранта.

In [ ]:
pulse_sizes = [50, 100, 200, 400]
pulse_grid = {}
print(f"{'C':>4} {'N':>4} {'E₁ upwind':>12} {'min upwind':>12} {'max upwind':>12} {'E₁ Л–В':>12} {'min Л–В':>12} {'max Л–В':>12}")
for C in stable_C:
    for N in pulse_sizes:
        run = stable_runs[C] if N == N_pulse else solve(N, C, 5.0, 'pulse')
        pulse_grid[N, C] = run
        a, b = run['upwind'], run['LW']
        print(f"{C:4.1f} {N:4d} {a['error'][-1]:12.4e} {a['minimum'][-1]:12.4e} {a['maximum'][-1]:12.5f} {b['error'][-1]:12.4e} {b['minimum'][-1]:12.4e} {b['maximum'][-1]:12.5f}")

fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True, sharey='row', constrained_layout=True)
dx_pulse = 1 / np.array(pulse_sizes)
for col, C in enumerate([0.5, 0.8]):
    for row, (key, ylabel) in enumerate([('error', 'E₁'), ('minimum', 'min u'), ('maximum', 'max u')]):
        ax = axes[row, col]
        for name in methods:
            values = [pulse_grid[N, C][name][key][-1] for N in pulse_sizes]
            ax.plot(dx_pulse, values, 'o-', color=colors[name], label=labels[name])
        ax.set_xscale('log')
        ax.set_xticks(dx_pulse, labels=['1/50', '1/100', '1/200', '1/400'])
        ax.xaxis.set_minor_locator(NullLocator())
        ax.set(xlabel='Δx', ylabel=ylabel)
        if row == 0:
            ax.set_yscale('log')
            ax.set_title(f'C = {C:g}, t = 5')
        else:
            ax.axhline(row-1, color='gray', linestyle=':', linewidth=1)
        ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
plt.show()

При сгущении сетки $E_1$ уменьшается, но первый и второй порядки, полученные на гладком профиле, здесь не сохраняются.
В решении по схеме против потока максимум приближается к единице.
Для схемы Лакса–Вендроффа интегральная ошибка меньше, однако выбросы не обязаны монотонно уменьшаться: область ошибки сужается, а её высота может сохраняться.
Например, при $C=0.8$, $N=400$ получены $E_1\approx0.0442$, минимум $-0.232$ и максимум $1.232$.

## случай C = 1

При $C=1$ обе формулы сводятся к

$$u_j^{n+1}=u_{j-1}^n.$$

За шаг точное решение также сдвигается на один узел. Поэтому при расчёте обеими схемами на временных слоях получается точный сдвиг начального массива, в том числе прямоугольного импульса.
В таблице для всех сеток получены $E_1=0$, минимум 0 и максимум 1.
Этот случай не показывает обычный порядок сходимости: здесь совпали перемещение за шаг и расстояние между узлами.
Если последний шаг сокращён, его $C_n<1$, и точный сдвиг на один узел на этом шаге уже не выполняется.

## минимум, максимум и ошибка во времени

Для устойчивых расчётов используем общие моменты $t=0,0.02,\ldots,5$ на сетке $N=200$.
Они совпадают с временными слоями всех трёх расчётов, поэтому интерполяция и дополнительные сокращённые шаги не нужны.

In [ ]:
common_times = np.arange(251)*0.02
fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True, sharey='row', constrained_layout=True)
metrics = [('minimum', 'min u'), ('maximum', 'max u'), ('error', 'E₁')]
for col, C in enumerate(stable_C):
    run = stable_runs[C]
    indices = np.rint(common_times / (C/N_pulse)).astype(int)
    for row, (key, ylabel) in enumerate(metrics):
        ax = axes[row, col]
        for name in methods:
            ax.plot(common_times, run[name][key][indices], color=colors[name],
                     linestyle=styles[name], label=labels[name])
        ax.set(xlabel='t', ylabel=ylabel)
        if row == 0:
            ax.set_title(f'C = {C:g}')
            ax.axhline(0, color='gray', linestyle=':', linewidth=1)
        elif row == 1:
            ax.axhline(1, color='gray', linestyle=':', linewidth=1)
        ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=7)
plt.show()

При $C=0.5$ размытие по схеме против потока сильнее, чем при $C=0.8$.
Для схемы Лакса–Вендроффа при обоих значениях $C$ видны выходы за физический диапазон, но быстрого роста их амплитуды со временем нет.
Это отличается от неустойчивости при нарушении условия CFL.

## неустойчивый расчёт

Для обеих схем условие устойчивости при $c>0$ имеет вид $0\leq C\leq1$.
Отдельно берём $C=1.1$, $N=100$ и рассчитываем перенос до $t=0.55$ — это 50 полных шагов.
На графиках минимума и максимума используем симметричную логарифмическую шкалу, чтобы сохранить знак значений.

In [ ]:
unstable = solve(100, 1.1, 0.55, 'pulse')
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for ax, (key, ylabel) in zip(axes, metrics):
    for name in methods:
        ax.plot(unstable['t'], unstable[name][key], color=colors[name],
                 linestyle=styles[name], label=labels[name])
    ax.set_yscale('symlog', linthresh=0.01 if key == 'error' else 1)
    ax.set(xlabel='t', ylabel=ylabel, title='C = 1.1, N = 100')
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
plt.show()
for name in methods:
    data = unstable[name]
    print(f"{labels[name]}: min = {data['minimum'][-1]:.3e}, max = {data['maximum'][-1]:.3e}, E₁ = {data['error'][-1]:.3e}")

При $C=1.1$ ошибка и экстремумы быстро растут по модулю. Уже к $t=0.55$ диапазон значений составляет примерно $[-956,957]$ для схемы против потока и $[-2.34\cdot10^6,2.34\cdot10^6]$ для схемы Лакса–Вендроффа при начальной высоте импульса 1.
Такие значения вызваны неустойчивостью, а не обычным размытием или ограниченными колебаниями около фронтов.

## анимация переноса

Показываем два оборота на сетке $N=200$ при $C=0.5,0.8,1$.
Во всех панелях одинаковы время и масштаб. Точный профиль и оба численных решения наложены друг на друга.
При переходе через $x=1$ импульс появляется у $x=0$.

In [ ]:
frame_times = np.arange(101)*0.02
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
artists = []
for ax, C in zip(axes, stable_C):
    run = stable_runs[C]
    lines = [plot_profile(ax, run, name, 0) for name in methods]
    reference_line, = ax.plot(*pulse_line(0), ':', color='black', linewidth=1.5, label='точное решение')
    artists.append((C, run, lines, reference_line))
    ax.set(xlabel='x', ylabel='u', xlim=(0, 1), ylim=(-0.35, 1.35), title=f'C = {C:g}, N = {N_pulse}')
    ax.grid(alpha=0.25)
clock = fig.suptitle('t = 0.00')
fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=3, fontsize=9)
fig.tight_layout(rect=(0, 0.09, 1, 0.94))


def update(frame):  # все три расчёта и точный профиль в один момент времени
    t = frame_times[frame]
    for C, run, lines, reference_line in artists:
        index = round(t / (C/N_pulse))
        for name, line in zip(methods, lines):
            u = run[name]['u'][index]
            line.set_ydata(np.r_[u, u[0]])
        reference_line.set_data(*pulse_line(t))
    clock.set_text(f't = {t:.2f}')
    return [clock] + [line for _, _, lines, reference in artists for line in lines+[reference]]


animation = FuncAnimation(fig, update, frames=len(frame_times), interval=40, blit=False)
with TemporaryDirectory() as folder:
    path = Path(folder) / 'advection.gif'
    animation.save(path, writer=PillowWriter(fps=25), dpi=90)
    gif_bytes = path.read_bytes()
plt.close(fig)
display(Image(data=gif_bytes, format='gif'))